# 04 · Train EfficientNet-B0 (+ optional B2)

**Checklist:**
- [ ] Notebook 03 complete — ResNet-50 checkpoint saved to Drive
- [ ] Runtime → **A100 GPU**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────
import os, sys

GITHUB_USER = 'musarashid49'
REPO_NAME   = 'Image-Classification-with-CNN'
REPO_DIR    = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ NOT FOUND')

In [ ]:
LOCAL_DATA_DIR   = '/content/data'
DRIVE_SPLIT_PATH = '/content/drive/MyDrive/pk_politicians_split'
DRIVE_RESULTS    = '/content/drive/MyDrive/pk_politicians_results'

In [ ]:
from src.utils import copy_dataset_from_drive, set_seed
from src.dataset import get_dataloaders

set_seed(42)
copy_dataset_from_drive(DRIVE_SPLIT_PATH, LOCAL_DATA_DIR)

loaders = get_dataloaders(
    os.path.join(LOCAL_DATA_DIR, 'train'),
    os.path.join(LOCAL_DATA_DIR, 'val'),
    os.path.join(LOCAL_DATA_DIR, 'test'),
)

In [ ]:
# ═══════════════════════════════════════════════
# EfficientNet-B0
# ═══════════════════════════════════════════════
from src.models import build_model
from src.train  import Trainer

model_b0 = build_model('efficientnet_b0', num_classes=16, dropout=0.4)

trainer_b0 = Trainer(
    model      = model_b0,
    loaders    = loaders,
    model_name = 'efficientnet_b0',
    config_overrides = {'num_epochs': 30, 'learning_rate': 1e-4}
)
history_b0 = trainer_b0.run()

In [ ]:
from src.evaluate import evaluate_model, plot_training_curves, plot_confusion_matrix, plot_misclassified

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plot_training_curves(history_b0, 'efficientnet_b0').show()

res_b0 = evaluate_model(model_b0, loaders['test'], device, model_name='efficientnet_b0')
plot_confusion_matrix(res_b0['y_true'], res_b0['y_pred'], model_name='efficientnet_b0').show()
fig = plot_misclassified(res_b0['images'], res_b0['y_true'], res_b0['y_pred'], model_name='efficientnet_b0')
if fig: fig.show()

In [ ]:
# ═══════════════════════════════════════════════
# (OPTIONAL) EfficientNet-B2  — uncomment to run
# ═══════════════════════════════════════════════

# loaders_b2 = get_dataloaders(
#     os.path.join(LOCAL_DATA_DIR, 'train'),
#     os.path.join(LOCAL_DATA_DIR, 'val'),
#     os.path.join(LOCAL_DATA_DIR, 'test'),
#     image_size=260,   # B2 native resolution
# )
# model_b2 = build_model('efficientnet_b2', num_classes=16, dropout=0.4)
# trainer_b2 = Trainer(model_b2, loaders_b2, 'efficientnet_b2',
#                      config_overrides={'num_epochs': 30, 'learning_rate': 1e-4})
# history_b2 = trainer_b2.run()
# res_b2 = evaluate_model(model_b2, loaders_b2['test'], device, model_name='efficientnet_b2')
# plot_confusion_matrix(res_b2['y_true'], res_b2['y_pred'], model_name='efficientnet_b2').show()

In [ ]:
from src.utils import ExperimentLogger, save_results_to_drive
logger = ExperimentLogger()
logger.log(
    model_name    = 'efficientnet_b0',
    test_accuracy = round(res_b0['accuracy'], 4),
    macro_f1      = round(res_b0['report']['macro avg']['f1-score'], 4),
    epochs_run    = len(history_b0['train_loss']),
    lr=1e-4, batch_size=32,
    notes = 'Full fine-tuning, dropout=0.4',
)
save_results_to_drive('results', DRIVE_RESULTS)
print('✓ Done.  Next → 05_evaluation_and_plots.ipynb')